In [ ]:
# /// script
# requires-python = ">=3.13"
# dependencies = [
#     "numpy",
#     "pyarrow",
# ]
# ///

In [24]:
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.csv
from pathlib import Path

input_file = "41586_2010_BFnature08973_MOESM239_ESM.pdb"

In [27]:
def model_lines_to_coord_arrays(lines):
    l_num = 0
    # print(lines)
    for line in lines.split('\n'):
        line_annot = line[0:6].strip()
        l_num += 1
        # print(line)
        # if l_num > 10:
        #     break
        # print(f"line_annot: {line_annot}")
        if line_annot == "HETATM" or line_annot == "ATOM":
            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])
            chrom = line[21:22]
            # coord = int(line[83:90])
            x_arr.append(x)
            y_arr.append(y)
            z_arr.append(z)
            chr_arr.append(chrom)
            # coord_arr.append(coord)
            # chr_arr.append("test")
            coord_arr.append(123)
            if l_num <= 1:
                print(f'x: {x}, y: {y}, z: {z}, chr: {chrom}')
    return [x_arr, y_arr, z_arr, chr_arr, coord_arr]

def arrays_to_table(x_arr, y_arr, z_arr, chrom_arr, coord_arr):
    x_nparr = np.array(x_arr)
    y_nparr = np.array(y_arr)
    z_nparr = np.array(z_arr)
    chrom_nparr = np.array(chrom_arr)
    coord_nparr = np.array(coord_arr)
    return pa.Table.from_arrays([x_nparr, y_nparr, z_nparr, chrom_nparr, coord_nparr], ["x", "y", "z", "chr", "coord"])

def write_to_file(outfile_path, table):
    with pa.OSFile(outfile_path, 'wb') as sink:
        with pa.ipc.RecordBatchFileWriter(sink, table.schema) as writer:
            writer.write_table(table)

out_folder = Path("out")
out_folder.mkdir(parents=True, exist_ok=True)

outputs_list_file = "out/out_file.txt"
with open(outputs_list_file, "w") as outputs_file:
    filepath = f"./{input_file}"
    
    with open(filepath, "r") as file:
        lines = file.read()

        x_arr = []
        y_arr = []
        z_arr = []
        chr_arr = []
        coord_arr = []
        [x_arr, y_arr, z_arr, chr_arr, coord_arr] = model_lines_to_coord_arrays(lines)
        table = arrays_to_table(x_arr, y_arr, z_arr, chr_arr, coord_arr)
    
        outputpath_parquet = f'out/yeast_model.parquet'
        print("Writing: " + outputpath_parquet)
        pq.write_table(table, outputpath_parquet, compression=None)
        
        outputpath_arrow = f'out/yeast_model.arrow'
        print("Writing: " + outputpath_arrow)
        write_to_file(outputpath_arrow, table)

        pa.csv.write_csv(table, "out/yeast_model.csv",
                 write_options=pa.csv.WriteOptions(include_header=True))
        
        outputs_file.write(f'yeast_model\n')

x: 77.365, y: 61.609, z: 15.448, chr: A
Writing: out/yeast_model.parquet
Writing: out/yeast_model.arrow
